# 05 — Gold: FactSales (Transacional) — Spark SQL

Grain: `(OrderID, ProductID)`.

**Técnica Spark SQL:** JOIN SCD2-aware + `MERGE INTO` para upsert idempotente.

A condição `ValidFrom <= OrderDate < ValidTo` garante que o FK aponta para a versão
do cliente/produto vigente *na data do pedido*, não a versão atual.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 05 FactSales")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:49:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


### ⚠️ O join "ingênuo" (sem SCD2-aware) produz resultados errados

Se fizéssemos um equi-join simples `ON o.CustomerID = dc.CustomerID AND dc.IsCurrent = true`,
estaríamos ligando cada linha de FactSales ao **endereço atual** do cliente, não ao endereço
que ele tinha na **data do pedido**.

Para Northwind (dados de 1996–1998 + simulação de mudança no LAB do notebook 04),
isso contamina análises geográficas: "receita por cidade" mostraria a cidade de hoje,
não a de quando o pedido foi feito.

**A correção:** `ValidFrom <= OrderDate < ValidTo` — filtro de intervalo temporal.  
A célula abaixo demonstra o problema com dados reais.


In [3]:
# Join INGÊNUO: usa versão ATUAL do cliente (IsCurrent=True)
wrong_n = spark.sql("""
    SELECT COUNT(*) AS n
    FROM bronze.order_details od
    JOIN bronze.orders o ON od.OrderID = o.OrderID
    JOIN gold.DimCustomer dc
      ON o.CustomerID = dc.CustomerID AND dc.IsCurrent = true
""").collect()[0]['n']

# Join CORRETO: versão vigente na data do pedido
correct_n = spark.sql("""
    SELECT COUNT(*) AS n
    FROM bronze.order_details od
    JOIN bronze.orders o ON od.OrderID = o.OrderID
    JOIN gold.DimCustomer dc
      ON  o.CustomerID = dc.CustomerID
      AND CAST(o.OrderDate AS DATE) >= dc.ValidFrom
      AND CAST(o.OrderDate AS DATE) <  dc.ValidTo
""").collect()[0]['n']

print(f"Linhas join ingênuo:  {wrong_n}")
print(f"Linhas join correto:  {correct_n}")

# Pedidos onde a City seria diferente
diff_n = spark.sql("""
    SELECT COUNT(*) AS n FROM (
        SELECT w.OrderID
        FROM (
            SELECT od.OrderID, dc.City AS City_wrong
            FROM bronze.order_details od
            JOIN bronze.orders o ON od.OrderID = o.OrderID
            JOIN gold.DimCustomer dc
              ON o.CustomerID = dc.CustomerID AND dc.IsCurrent = true
        ) w
        JOIN (
            SELECT od.OrderID, dc.City AS City_correct
            FROM bronze.order_details od
            JOIN bronze.orders o ON od.OrderID = o.OrderID
            JOIN gold.DimCustomer dc
              ON  o.CustomerID = dc.CustomerID
              AND CAST(o.OrderDate AS DATE) >= dc.ValidFrom
              AND CAST(o.OrderDate AS DATE) <  dc.ValidTo
        ) c ON w.OrderID = c.OrderID
        WHERE w.City_wrong != c.City_correct
    )
""").collect()[0]['n']
print(f"Pedidos com City diferente: {diff_n} (= impacto real do bug SCD2)")


Linhas join ingênuo:  2155
Linhas join correto:  2155


Pedidos com City diferente: 0 (= impacto real do bug SCD2)


In [4]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW src_fact_sales AS
    SELECT
        ABS(HASH(od.OrderID, od.ProductID))                             AS SalesSK,
        CAST(DATE_FORMAT(CAST(o.OrderDate AS DATE), 'yyyyMMdd') AS INT) AS OrderDateKey,
        dc.CustomerSK,
        dp.ProductSK,
        de.EmployeeSK,
        ds.ShipperSK,
        od.OrderID,
        od.ProductID,
        od.UnitPrice,
        od.Quantity,
        od.Discount,
        od.UnitPrice * od.Quantity                          AS GrossRevenue,
        od.UnitPrice * od.Quantity * (1.0 - od.Discount)   AS NetRevenue,
        current_timestamp()                                 AS LoadTimestamp
    FROM bronze.order_details od
    JOIN bronze.orders o ON od.OrderID = o.OrderID
    -- SCD2-aware: versão vigente na data do pedido
    JOIN gold.DimCustomer dc
      ON  o.CustomerID = dc.CustomerID
      AND CAST(o.OrderDate AS DATE) >= dc.ValidFrom
      AND CAST(o.OrderDate AS DATE) <  dc.ValidTo
    JOIN gold.DimProduct dp
      ON  od.ProductID = dp.ProductID
      AND CAST(o.OrderDate AS DATE) >= dp.ValidFrom
      AND CAST(o.OrderDate AS DATE) <  dp.ValidTo
    JOIN gold.DimEmployee de ON o.EmployeeID = de.EmployeeID
    JOIN gold.DimShipper  ds ON o.ShipVia    = ds.ShipperID
""")

n = spark.sql("SELECT COUNT(*) AS n FROM src_fact_sales").collect()[0]["n"]
print(f"Source FactSales: {n} linhas (esperado: 2155)")
spark.sql("SELECT * FROM src_fact_sales LIMIT 3").show()

Source FactSales: 2155 linhas (esperado: 2155)


+----------+------------+----------+----------+----------+----------+-------+---------+---------+--------+--------+------------+----------+--------------------+
|   SalesSK|OrderDateKey|CustomerSK| ProductSK|EmployeeSK| ShipperSK|OrderID|ProductID|UnitPrice|Quantity|Discount|GrossRevenue|NetRevenue|       LoadTimestamp|
+----------+------------+----------+----------+----------+----------+-------+---------+---------+--------+--------+------------+----------+--------------------+
|1057924303|    19960704|1235774628|1769050788|1023896466|1823081949|  10248|       11|     14.0|      12|     0.0|       168.0|     168.0|2026-03-29 00:50:...|
| 374317602|    19960704|1235774628|1607902733|1023896466|1823081949|  10248|       42|      9.8|      10|     0.0|        98.0|      98.0|2026-03-29 00:50:...|
| 689028109|    19960704|1235774628|  63553776|1023896466|1823081949|  10248|       72|     34.8|       5|     0.0|       174.0|     174.0|2026-03-29 00:50:...|
+----------+------------+---------

In [5]:
spark.sql("""
    MERGE INTO gold.FactSales AS tgt
    USING src_fact_sales AS src
    ON tgt.OrderID = src.OrderID AND tgt.ProductID = src.ProductID
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

n = spark.sql("SELECT COUNT(*) AS n FROM gold.FactSales").collect()[0]["n"]
print(f"FactSales após MERGE: {n} linhas (esperado: 2155)")

FactSales após MERGE: 2155 linhas (esperado: 2155)


In [6]:
print("1. Contagem total:")
n_fact   = spark.sql("SELECT COUNT(*) AS n FROM gold.FactSales").collect()[0]["n"]
n_bronze = spark.sql("SELECT COUNT(*) AS n FROM bronze.order_details").collect()[0]["n"]
print(f"   FactSales={n_fact}, bronze.order_details={n_bronze}, Match={n_fact == n_bronze}")

print("\n2. Receita total:")
spark.sql("""
    SELECT ROUND(SUM(GrossRevenue), 2) AS GrossRevenue,
           ROUND(SUM(NetRevenue), 2)   AS NetRevenue,
           ROUND(SUM(GrossRevenue) - SUM(NetRevenue), 2) AS Desconto
    FROM gold.FactSales
""").show()

print("3. Órfãos:")
for sk, dim in [("CustomerSK", "gold.DimCustomer"), ("ProductSK", "gold.DimProduct")]:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM gold.FactSales fs "
                  f"WHERE NOT EXISTS (SELECT 1 FROM {dim} d WHERE d.{sk} = fs.{sk})"
                  ).collect()[0]["n"]
    print(f"   Órfãos por {sk}: {n} (esperado: 0)")

print("\n4. Top 5 por NetRevenue:")
spark.sql("""
    SELECT dp.ProductName, dp.CategoryName,
           ROUND(SUM(fs.NetRevenue), 2) AS NetRevenue
    FROM gold.FactSales fs
    JOIN gold.DimProduct dp ON dp.ProductSK = fs.ProductSK
    GROUP BY dp.ProductName, dp.CategoryName
    ORDER BY NetRevenue DESC LIMIT 5
""").show()

1. Contagem total:


   FactSales=2155, bronze.order_details=2155, Match=True

2. Receita total:


+------------+----------+--------+
|GrossRevenue|NetRevenue|Desconto|
+------------+----------+--------+
|  1354458.59|1265793.04|88665.55|
+------------+----------+--------+

3. Órfãos:


   Órfãos por CustomerSK: 0 (esperado: 0)


   Órfãos por ProductSK: 0 (esperado: 0)

4. Top 5 por NetRevenue:


+--------------------+--------------+----------+
|         ProductName|  CategoryName|NetRevenue|
+--------------------+--------------+----------+
|       Côte de Blaye|     Beverages| 141396.73|
|Thüringer Rostbra...|  Meat/Poultry|  80368.67|
|Raclette Courdavault|Dairy Products|   71155.7|
|      Tarte au sucre|   Confections|  47234.97|
|   Camembert Pierrot|Dairy Products|  46825.48|
+--------------------+--------------+----------+

